<a href="https://colab.research.google.com/github/miguelgorr/Projects/blob/main/U87_MS_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, numpy as np, os, requests, collections
from scipy import stats

In [ ]:
#Identify the Characteristic Processes at each time interval

In [2]:
U87_xlsm_files = [
                  'U87_D12_R3.xlsm',
                  'U87_D3_R4.xlsm',
                  'U87_D6_R4.xlsm',
                  'U87_D9_R1.xlsm'
                  ]

In [5]:
#Convert .xlsm into pandas df and organize in dict with sample name as key and pandas df as value

mydrive_path = '/content/drive/MyDrive/Gorriceta Research 2026/Reeck Lab/U87_MS_Analysis/U87_Raw_Mass_Spec/'
U87_df_dict = {}

for file in U87_xlsm_files:

  file_path = f'{mydrive_path}{file}'
  df = pd.read_excel(file_path)
  U87_df_dict[file] = df
  print(f'Successfully uploaded {file}')

Successfully uploaded U87_D12_R3.xlsm
Successfully uploaded U87_D3_R4.xlsm
Successfully uploaded U87_D6_R4.xlsm
Successfully uploaded U87_D9_R1.xlsm


In [6]:
#Use this to see what the column names are for the dfs
def view_colnames(dict):
  for key,item in dict.items():
    if isinstance(item, pd.DataFrame):
      print(item.columns)
    else:
      print(f'{key} is not a pandas DataFrame object')

In [7]:
view_colnames(U87_df_dict)

Index(['Checked', 'Master', 'Accession', 'Description', 'Coverage [%]',
       '# Peptides', '# PSMs', '# Unique Peptides', '# AAs', 'MW [kDa]',
       'calc. pI', 'Score Sequest HT: Sequest HT',
       '# Peptides (by Search Engine): Sequest HT', 'Biological Process',
       'Cellular Component', 'Molecular Function', 'Pfam IDs',
       'Entrez Gene ID', 'Ensembl Gene ID', 'Gene Symbol', 'Gene ID',
       'Reactome Pathways', 'WikiPathways', '# Protein Pathway Groups',
       'Found in Sample: F4: Sample', '# Protein Groups'],
      dtype='object')
Index(['Checked', 'Master', 'Accession', 'Description', 'Coverage [%]',
       '# Peptides', '# PSMs', '# Unique Peptides', '# AAs', 'MW [kDa]',
       'calc. pI', 'Score Sequest HT: Sequest HT',
       '# Peptides (by Search Engine): Sequest HT', 'Biological Process',
       'Cellular Component', 'Molecular Function', 'Pfam IDs',
       'Entrez Gene ID', 'Ensembl Gene ID', 'Gene Symbol', 'Gene ID',
       'WikiPathways', 'Reactome Pathways

In [ ]:
#Create filtering criteria

#Find the proportion of the total PSMs that each protein accounts for
def majority_psms(dict_of_samples):
  for key,item in dict_of_samples.items():
    list_of_proportions = []
    df_length = len(item)
    psms_series = item['# PSMs']
    for entry in psms_series:
      proportion_of_total = entry / df_length
      item['Proportion Total PSMs'] = proportion_of_total

#High PSMs with low coverage
def high_psms_low_cov(dict_of_samples):
  cov_psms_dict = {}
  for key, item in dict_of_samples.items():
    #Check if you have a dictionary of pd DataFrames and they have the right columns
    if isinstance(item, pd.DataFrame) and all(col in item.columns for col in ['Gene Symbol', 'Coverage [%]', '# PSMs']):
      cov_psms_df = item[['Gene Symbol','Coverage [%]', '# PSMs']]


      #Check for coverage <50%
      cov_psms_df = cov_psms_df[~(cov_psms_df['Coverage [%]'] > 50)]

      #Filter by PSMs > trimmed mean (trimming the top and bottom 5%)
      trimmed_mean_val = stats.trim_mean(cov_psms_df['# PSMs'], proportiontocut=0.10)
      print(f'Trimmed Mean: {trimmed_mean_val}')
      cov_psms_df = cov_psms_df[~(cov_psms_df['# PSMs'] < trimmed_mean_val)]

      #Sort by # PSMs highest to lowest
      cov_psms_df = cov_psms_df.sort_values(by='# PSMs', ascending=False)


      cov_psms_dict[key] = cov_psms_df

      #Save to Google Drive Folder
  output_path = '/content/drive/MyDrive/Gorriceta Research 2026/Reeck Lab/U87_MS_Analysis/U87_Analysis_Results'
  os.makedirs(output_path, exist_ok=True)

  for filename, df in cov_psms_dict.items():
    clean_name = filename.removesuffix('.xlsm')
    save_file_path = f'{output_path}/{clean_name}_filtered.xlsx'
    df.to_excel(save_file_path, index=False)


  return cov_psms_dict




In [ ]:
high_psms_low_cov(U87_df_dict)

Trimmed Mean: 5.908242197919445
Trimmed Mean: 5.571238348868175
Trimmed Mean: 5.649553571428571
Trimmed Mean: 5.541427660930529


{'U87_D12_R3.xlsm':      Gene Symbol  Coverage [%]  # PSMs
 280        AHNAK            48     212
 448      DYNC1H1            40     194
 240         FLNA            50     168
 4060         TTN             5     136
 284         FASN            48     121
 ...          ...           ...     ...
 4716        SBF2             3       6
 4691      LRRC41             3       6
 4688       YLPM1             3       6
 4670     SHROOM4             3       6
 921       NSFL1C            24       6
 
 [2183 rows x 3 columns],
 'U87_D3_R4.xlsm':      Gene Symbol  Coverage [%]  # PSMs
 4          POTEE            18     189
 10       DYNC1H1            41     181
 19         AHNAK            42     157
 44           TTN             4     103
 30          CHD1             5      93
 ...          ...           ...     ...
 2119    SH3PXD2B             7       6
 2123      LRRIQ4            12       6
 2137      CFAP74             6       6
 2145       EPHA2             7       6
 2164       NCO

In [ ]:
x

{'U87_D12_R3.xlsm':      Gene Symbol  Coverage [%]  # PSMs
 280        AHNAK            48     212
 448      DYNC1H1            40     194
 240         FLNA            50     168
 4060         TTN             5     136
 284         FASN            48     121
 ...          ...           ...     ...
 4716        SBF2             3       6
 4691      LRRC41             3       6
 4688       YLPM1             3       6
 4670     SHROOM4             3       6
 921       NSFL1C            24       6
 
 [2183 rows x 3 columns],
 'U87_D3_R4.xlsm':      Gene Symbol  Coverage [%]  # PSMs
 4          POTEE            18     189
 10       DYNC1H1            41     181
 19         AHNAK            42     157
 44           TTN             4     103
 30          CHD1             5      93
 ...          ...           ...     ...
 2119    SH3PXD2B             7       6
 2123      LRRIQ4            12       6
 2137      CFAP74             6       6
 2145       EPHA2             7       6
 2164       NCO

In [ ]:
first_sample = U87_df_dict.get('U87_D12_R3.xlsm')
print(first_sample['Proportion Total PSMs'].max())

0.0006092607636068237


In [ ]:
#Check for max PSM for each day
top_10_dict = {}
gene_id_dict = {}

for df_name, df in U87_df_dict.items():
  sorted_psm_df = df.sort_values(by='# PSMs', ascending=False)
  top_10_df = sorted_psm_df.iloc[:10]
  top_10_dict[df_name] = top_10_df


  #Get a dict of the Gene IDs
  gene_id_dict[df_name] = top_10_df[['Gene Symbol', 'Reactome Pathways']]


In [ ]:
for sample, df in gene_id_dict.items():
  print(f'-----------{sample}-----------')
  display(df)

-----------U87_D12_R3.xlsm-----------


,Gene Symbol,Reactome Pathways
9,ENO1,Glycolysis; Gluconeogenesis; Manipulation of h...
28,PKM,Negative Regulation of CDH1 Gene Transcription...
3,GAPDH,Glycolysis; Gluconeogenesis
2,ACTB,Translocation of SLC2A4 (GLUT4) to the plasma ...
46,ENO2,Glycolysis; Gluconeogenesis
1,PGK1,Glycolysis; Gluconeogenesis; MTOR signalling; ...
0,LDHA,Pyruvate metabolism; Regulation of pyruvate me...
26,HSPD1,Mitochondrial unfolded protein response (UPRmt...
12,ALDOA,Glycolysis; Gluconeogenesis; Neutrophil degran...
65,EEF1A1,Neutrophil degranulation; Chaperone Mediated A...


-----------U87_D3_R4.xlsm-----------


,Gene Symbol,Reactome Pathways
0,ENO1,Glycolysis; Gluconeogenesis; Manipulation of h...
1,ACTB,Regulation of endogenous retroelements by Piwi...
2,GAPDH,Glycolysis; Gluconeogenesis
3,PKM,Neutrophil degranulation; Negative Regulation ...
5,ENO2,Glycolysis; Gluconeogenesis
6,MYH9,Translocation of SLC2A4 (GLUT4) to the plasma ...
4,POTEE,RHOF GTPase cycle
10,DYNC1H1,HSP90 chaperone cycle for steroid hormone rece...
9,EEF1A1,Neutrophil degranulation; Chaperone Mediated A...
12,VIM,Aggrephagy; Interleukin-4 and Interleukin-13 s...


-----------U87_D6_R4.xlsm-----------


,Gene Symbol,Reactome Pathways
0,ENO1,Glycolysis; Gluconeogenesis; Manipulation of h...
2,PKM,Neutrophil degranulation; Negative Regulation ...
1,GAPDH,Glycolysis; Gluconeogenesis
3,ACTB,Regulation of endogenous retroelements by Piwi...
4,ENO2,Glycolysis; Gluconeogenesis
5,ENO3,Glycolysis; Gluconeogenesis
6,EEF1A1,Neutrophil degranulation; Chaperone Mediated A...
7,VIM,Aggrephagy; Interleukin-4 and Interleukin-13 s...
9,EEF2,Neutrophil degranulation; Protein methylation;...
8,MYH9,Translocation of SLC2A4 (GLUT4) to the plasma ...


-----------U87_D9_R1.xlsm-----------


,Gene Symbol,Reactome Pathways
6,ENO1,Glycolysis; Gluconeogenesis; Manipulation of h...
25,PKM,Neutrophil degranulation; Negative Regulation ...
1,ACTB,UCH proteinases; Activation of STAT3 by cadher...
2,GAPDH,Glycolysis; Gluconeogenesis
34,ENO2,Glycolysis; Gluconeogenesis
5,PGK1,Glycolysis; Gluconeogenesis; MTOR signalling; ...
204,AHNAK,NaN
129,MYH9,Translocation of SLC2A4 (GLUT4) to the plasma ...
12,VIM,Aggrephagy; Interleukin-4 and Interleukin-13 s...
58,EEF1A1,Neutrophil degranulation; Chaperone Mediated A...


In [ ]:

for files, gene_data in gene_id_dict.items():

 #Break up the 'Molecular Function' string into its individual functions
 gene_data['Reactome'] = (
    gene_data['Reactome Pathways']
    .fillna('')
    .apply(lambda x: [item.strip() for item in x.split(';')])

  )


/tmp/ipykernel_1804/3559710813.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_data['Reactome'] = (
/tmp/ipykernel_1804/3559710813.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_data['Reactome'] = (
/tmp/ipykernel_1804/3559710813.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#

In [ ]:

count_dict = {}
master_list = []

for sample, df in gene_id_dict.items():
  temp_list = []


  for item in df['Reactome Pathways']:
    temp_list.extend(item)

  print(temp_list)

  master_list.extend(temp_list)

  represented_mol_fxns = set(master_list)

  for word in represented_mol_fxns:
    count = master_list.count(word)
    function_count = []




['G', 'l', 'y', 'c', 'o', 'l', 'y', 's', 'i', 's', ';', ' ', 'G', 'l', 'u', 'c', 'o', 'n', 'e', 'o', 'g', 'e', 'n', 'e', 's', 'i', 's', ';', ' ', 'M', 'a', 'n', 'i', 'p', 'u', 'l', 'a', 't', 'i', 'o', 'n', ' ', 'o', 'f', ' ', 'h', 'o', 's', 't', ' ', 'e', 'n', 'e', 'r', 'g', 'y', ' ', 'm', 'e', 't', 'a', 'b', 'o', 'l', 'i', 's', 'm', 'N', 'e', 'g', 'a', 't', 'i', 'v', 'e', ' ', 'R', 'e', 'g', 'u', 'l', 'a', 't', 'i', 'o', 'n', ' ', 'o', 'f', ' ', 'C', 'D', 'H', '1', ' ', 'G', 'e', 'n', 'e', ' ', 'T', 'r', 'a', 'n', 's', 'c', 'r', 'i', 'p', 't', 'i', 'o', 'n', ';', ' ', 'N', 'e', 'u', 't', 'r', 'o', 'p', 'h', 'i', 'l', ' ', 'd', 'e', 'g', 'r', 'a', 'n', 'u', 'l', 'a', 't', 'i', 'o', 'n', 'G', 'l', 'y', 'c', 'o', 'l', 'y', 's', 'i', 's', ';', ' ', 'G', 'l', 'u', 'c', 'o', 'n', 'e', 'o', 'g', 'e', 'n', 'e', 's', 'i', 's', 'T', 'r', 'a', 'n', 's', 'l', 'o', 'c', 'a', 't', 'i', 'o', 'n', ' ', 'o', 'f', ' ', 'S', 'L', 'C', '2', 'A', '4', ' ', '(', 'G', 'L', 'U', 'T', '4', ')', ' ', 't', 'o',

TypeError: 'float' object is not iterable

In [ ]:
function_count

[]

nucleic acid binding activity: 17
other molecular function: 40
cytoskeletal activity: 18
kinase activity: 6
enzyme regulator activity: 13
signal transduction activity or receptor binding: 3
translation activity: 1


In [ ]:
final_count_dict = {}

